# Testes de indicadores: warehouse × origem

Cada indicador da diretoria é calculado duas vezes: no **warehouse** (star schema) e direto na **origem** (o SQL Server transacional, com as regras de negócio aplicadas à mão: tipo Venda, não cancelado, data do ganho pela trilha de fases). A diferença tem de ser zero. Depois, os **ritos executivos** (mensal, semestral, anual) e as leituras de premiação (vendedores e parceiros) desenhados a partir do warehouse.

> Reprodutível: `uv run notebooks-modelo`.

In [1]:
import pandas as pd, pyodbc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from interiores_fictoria.config import conexao_origem, conexao_warehouse

pd.set_option("display.max_columns", 40); pd.set_option("display.width", 180)
dw = pyodbc.connect(conexao_warehouse(), timeout=15)
origem = pyodbc.connect(conexao_origem(), timeout=15)

from decimal import Decimal

def q(sql, cx=dw):
    cur = cx.cursor(); cur.execute(sql)
    cols = [d[0] for d in cur.description]
    linhas = [tuple(float(v) if isinstance(v, Decimal) else v for v in r) for r in cur.fetchall()]
    return pd.DataFrame.from_records(linhas, columns=cols)

print("warehouse:", q("SELECT DB_NAME() AS banco").iloc[0, 0], "| origem:", q("SELECT DB_NAME() AS banco", origem).iloc[0, 0])


warehouse: dw_fictoria | origem: db_fictoria


In [2]:
def comparar(nome, sql_dw, sql_origem):
    a = q(sql_dw).set_index("chave")["valor"].astype(float)
    b = q(sql_origem, origem).set_index("chave")["valor"].astype(float)
    df = pd.concat([a.rename("warehouse"), b.rename("origem")], axis=1).fillna(0.0).sort_index()
    df["diferenca"] = (df["warehouse"] - df["origem"]).round(2)
    df["ok"] = df["diferenca"].abs() < 0.01
    print(f"{'OK   ' if df['ok'].all() else 'FALHA'}  {nome}")
    return df


## 1. Indicadores: warehouse × origem

In [3]:
display(comparar('Orçamentos de venda por ano de cadastro (quantidade)', """SELECT ano AS chave, COUNT(*) AS valor FROM dw.ft_orcamento GROUP BY ano""", """SELECT YEAR(o.dt_cadastro) AS chave, COUNT(*) AS valor FROM comercial.orcamento o WHERE o.tipo_orcamento_id = 1 AND o.dt_cancelou IS NULL GROUP BY YEAR(o.dt_cadastro)"""))

OK     Orçamentos de venda por ano de cadastro (quantidade)


,warehouse,origem,diferenca,ok
chave,,,,
2021,1486.0,1486.0,0.0,True
2022,1719.0,1719.0,0.0,True
2023,2333.0,2333.0,0.0,True
2024,2571.0,2571.0,0.0,True
2025,3517.0,3517.0,0.0,True
2026,2554.0,2554.0,0.0,True


In [4]:
display(comparar('Total bruto orçado por ano de cadastro', """SELECT ano AS chave, SUM(vl_bruto) AS valor FROM dw.ft_orcamento GROUP BY ano""", """SELECT YEAR(o.dt_cadastro) AS chave, SUM(o.vl_total_bruto) AS valor FROM comercial.orcamento o WHERE o.tipo_orcamento_id = 1 AND o.dt_cancelou IS NULL GROUP BY YEAR(o.dt_cadastro)"""))

OK     Total bruto orçado por ano de cadastro


,warehouse,origem,diferenca,ok
chave,,,,
2021,1.903715e+08,1.903715e+08,0.0,True
2022,2.260058e+08,2.260058e+08,0.0,True
2023,3.561202e+08,3.561202e+08,0.0,True
2024,3.871462e+08,3.871462e+08,0.0,True
2025,5.852997e+08,5.852997e+08,0.0,True
2026,4.519252e+08,4.519252e+08,0.0,True


In [5]:
display(comparar('Vendas por ano do ganho (quantidade)', """SELECT ano AS chave, SUM(qtd_vendas) AS valor FROM dw.ft_venda GROUP BY ano""", """SELECT YEAR(g.dt_ganho) AS chave, COUNT(*) AS valor FROM comercial.orcamento o JOIN (SELECT orcamento_id, MIN(dt_entrada) AS dt_ganho FROM comercial.orcamento_fase_hist
                       WHERE fase_id = 6 GROUP BY orcamento_id) g ON g.orcamento_id = o.id WHERE o.tipo_orcamento_id = 1 AND o.dt_cancelou IS NULL AND o.fase_id = 6 GROUP BY YEAR(g.dt_ganho)"""))

OK     Vendas por ano do ganho (quantidade)


,warehouse,origem,diferenca,ok
chave,,,,
2021,123.0,123.0,0.0,True
2022,241.0,241.0,0.0,True
2023,419.0,419.0,0.0,True
2024,629.0,629.0,0.0,True
2025,1039.0,1039.0,0.0,True
2026,953.0,953.0,0.0,True


In [6]:
display(comparar('Vendas por ano do ganho (líquido)', """SELECT ano AS chave, SUM(vl_liquido) AS valor FROM dw.ft_venda GROUP BY ano""", """SELECT YEAR(g.dt_ganho) AS chave, SUM(o.vl_total_liquido) AS valor FROM comercial.orcamento o JOIN (SELECT orcamento_id, MIN(dt_entrada) AS dt_ganho FROM comercial.orcamento_fase_hist
                       WHERE fase_id = 6 GROUP BY orcamento_id) g ON g.orcamento_id = o.id WHERE o.tipo_orcamento_id = 1 AND o.dt_cancelou IS NULL AND o.fase_id = 6 GROUP BY YEAR(g.dt_ganho)"""))

OK     Vendas por ano do ganho (líquido)


,warehouse,origem,diferenca,ok
chave,,,,
2021,1.243944e+07,1.243944e+07,0.0,True
2022,2.422492e+07,2.422492e+07,0.0,True
2023,4.699141e+07,4.699141e+07,0.0,True
2024,7.085745e+07,7.085745e+07,0.0,True
2025,1.309223e+08,1.309223e+08,0.0,True
2026,1.364132e+08,1.364132e+08,0.0,True


In [7]:
display(comparar('Comissões apuradas por ano de competência', """SELECT ano AS chave, SUM(vl_comissao) AS valor FROM dw.ft_comissao GROUP BY ano""", """SELECT CAST(LEFT(c.competencia, 4) AS int) AS chave, SUM(c.vl_comissao) AS valor FROM financeiro.comissao c JOIN comercial.orcamento o ON o.id = c.orcamento_id WHERE o.tipo_orcamento_id = 1 AND o.dt_cancelou IS NULL GROUP BY LEFT(c.competencia, 4)"""))

OK     Comissões apuradas por ano de competência


,warehouse,origem,diferenca,ok
chave,,,,
2021,185570.01,185570.01,0.0,True
2022,401067.59,401067.59,0.0,True
2023,1085197.48,1085197.48,0.0,True
2024,1824911.08,1824911.08,0.0,True
2025,3590913.25,3590913.25,0.0,True
2026,3838200.68,3838200.68,0.0,True


In [8]:
display(comparar('Maior venda por ano (líquido)', """SELECT ano AS chave, MAX(vl_liquido) AS valor FROM dw.ft_venda GROUP BY ano""", """SELECT YEAR(g.dt_ganho) AS chave, MAX(o.vl_total_liquido) AS valor FROM comercial.orcamento o JOIN (SELECT orcamento_id, MIN(dt_entrada) AS dt_ganho FROM comercial.orcamento_fase_hist
                       WHERE fase_id = 6 GROUP BY orcamento_id) g ON g.orcamento_id = o.id WHERE o.tipo_orcamento_id = 1 AND o.dt_cancelou IS NULL AND o.fase_id = 6 GROUP BY YEAR(g.dt_ganho)"""))

OK     Maior venda por ano (líquido)


,warehouse,origem,diferenca,ok
chave,,,,
2021,776635.06,776635.06,0.0,True
2022,1326062.28,1326062.28,0.0,True
2023,1294194.50,1294194.50,0.0,True
2024,1939807.13,1939807.13,0.0,True
2025,3085417.77,3085417.77,0.0,True
2026,2565298.54,2565298.54,0.0,True


In [9]:
display(comparar('Situação atual do funil (abertos, ganhos, perdidos)', """SELECT grupo_fase AS chave, COUNT(*) AS valor FROM dw.ft_orcamento GROUP BY grupo_fase""", """SELECT f.grupo AS chave, COUNT(*) AS valor FROM comercial.orcamento o JOIN cadastro.fase f ON f.id = o.fase_id WHERE o.tipo_orcamento_id = 1 AND o.dt_cancelou IS NULL GROUP BY f.grupo"""))

OK     Situação atual do funil (abertos, ganhos, perdidos)


,warehouse,origem,diferenca,ok
chave,,,,
ABERTO,893.0,893.0,0.0,True
GANHO,3404.0,3404.0,0.0,True
PERDIDO,9883.0,9883.0,0.0,True


In [10]:
display(comparar('Orçamentos por grupo de canal (outras origens)', """SELECT c.grupo AS chave, COUNT(*) AS valor FROM dw.ft_orcamento o JOIN dw.dim_canal c ON c.sk_canal = o.sk_canal GROUP BY c.grupo""", """SELECT oc.grupo AS chave, COUNT(*) AS valor FROM comercial.orcamento o JOIN cadastro.origem_contato oc ON oc.id = o.origem_contato_id WHERE o.tipo_orcamento_id = 1 AND o.dt_cancelou IS NULL GROUP BY oc.grupo"""))

OK     Orçamentos por grupo de canal (outras origens)


,warehouse,origem,diferenca,ok
chave,,,,
ARQUITETOS,3092.0,3092.0,0.0,True
CANAL_PROPRIO,6882.0,6882.0,0.0,True
CONSTRUTORAS,442.0,442.0,0.0,True
INDICACAO_CLIENTE,3290.0,3290.0,0.0,True
OUTROS,474.0,474.0,0.0,True


In [11]:
display(comparar('Vendas por vendedor (líquido, top por id)', """SELECT sk_vendedor AS chave, SUM(vl_liquido) AS valor FROM dw.ft_venda GROUP BY sk_vendedor""", """SELECT o.vendedor_id AS chave, SUM(o.vl_total_liquido) AS valor FROM comercial.orcamento o WHERE o.tipo_orcamento_id = 1 AND o.dt_cancelou IS NULL AND o.fase_id = 6 GROUP BY o.vendedor_id"""))

OK     Vendas por vendedor (líquido, top por id)


,warehouse,origem,diferenca,ok
chave,,,,
1,17694092.80,17694092.80,0.0,True
2,42728774.71,42728774.71,0.0,True
3,4433818.58,4433818.58,0.0,True
4,34429700.69,34429700.69,0.0,True
5,19607975.96,19607975.96,0.0,True
6,7240887.93,7240887.93,0.0,True
7,5750263.82,5750263.82,0.0,True
8,17692354.81,17692354.81,0.0,True
9,22508603.14,22508603.14,0.0,True


In [12]:
display(comparar('Parcelas vencidas sem pagamento (quantidade) por ano de vencimento', """SELECT ano AS chave, SUM(fl_vencida_sem_pagamento) AS valor FROM dw.ft_parcela GROUP BY ano""", """SELECT YEAR(p.dt_vencimento) AS chave,
                   SUM(CASE WHEN p.fl_pago = 0 AND p.dt_cancelamento IS NULL AND p.dt_vencimento < tt.t
                            THEN 1 ELSE 0 END) AS valor
            FROM financeiro.parcela p
            JOIN financeiro.recebimento r ON r.id = p.recebimento_id
            JOIN comercial.orcamento o ON o.id = r.orcamento_id
            CROSS JOIN (SELECT CAST(MAX(dt_cadastro) AS date) AS t FROM comercial.orcamento) tt
            WHERE o.tipo_orcamento_id = 1 AND o.dt_cancelou IS NULL AND p.dt_cancelamento IS NULL GROUP BY YEAR(p.dt_vencimento)"""))

OK     Parcelas vencidas sem pagamento (quantidade) por ano de vencimento


,warehouse,origem,diferenca,ok
chave,,,,
2021,14.0,14.0,0.0,True
2022,49.0,49.0,0.0,True
2023,69.0,69.0,0.0,True
2024,106.0,106.0,0.0,True
2025,178.0,178.0,0.0,True
2026,249.0,249.0,0.0,True
2027,0.0,0.0,0.0,True


**Nota Técnica**

- **Observado:** 10 de 10 indicadores idênticos entre warehouse e origem, em todas as chaves (anos, grupos de fase, grupos de canal, vendedores).
- **Por que importa:** é a prova de que bronze, silver e gold não perderam nem inventaram nada: o número que a diretoria vê no dashboard é o mesmo que uma consulta cuidadosa no sistema devolveria, sem pesar na produção.
- **Ação:** qualquer divergência futura aparece aqui antes de chegar a um relatório; este notebook é a régua de aceite dos indicadores.

## 2. Os ritos executivos, lidos do warehouse

In [13]:
# Rito MENSAL: agosto de 2026 contra agosto de 2025 e de 2024
mensal = q("""
SELECT c.ano, SUM(v.qtd_vendas) AS vendas, SUM(v.vl_liquido) AS liquido, SUM(v.vl_comissao_total) AS comissoes
FROM dw.ft_venda v JOIN dw.dim_calendario c ON c.sk_data = v.sk_data_ganho
WHERE c.mes = 8 AND c.ano IN (2024, 2025, 2026) GROUP BY c.ano ORDER BY c.ano""")
display(mensal)
# Rito SEMESTRAL: janeiro a julho, três anos
semestral = q("""
SELECT c.ano, COUNT(*) AS orcamentos, SUM(o.fl_ganho) AS ganhos, SUM(o.fl_perdido) AS perdidos, SUM(o.fl_aberto) AS abertos,
       CAST(100.0 * SUM(o.fl_ganho) / NULLIF(SUM(o.fl_fechado), 0) AS decimal(5,1)) AS conversao_pct
FROM dw.ft_orcamento o JOIN dw.dim_calendario c ON c.sk_data = o.sk_data_cadastro
WHERE c.mes BETWEEN 1 AND 7 AND c.ano IN (2024, 2025, 2026) GROUP BY c.ano ORDER BY c.ano""")
display(semestral)
# Rito ANUAL: três anos fechados
anual = q("""
SELECT c.ano, SUM(v.qtd_vendas) AS vendas, SUM(v.vl_liquido) AS liquido, SUM(v.vl_bruto) AS bruto,
       CAST(SUM(v.vl_liquido) / NULLIF(SUM(v.vl_bruto), 0) AS decimal(5,3)) AS margem_liquida,
       CAST(SUM(v.vl_margem_custo) / NULLIF(SUM(v.vl_liquido), 0) AS decimal(5,3)) AS margem_sobre_custo,
       SUM(v.vl_comissao_total) AS comissoes
FROM dw.ft_venda v JOIN dw.dim_calendario c ON c.sk_data = v.sk_data_ganho
WHERE c.ano IN (2023, 2024, 2025) GROUP BY c.ano ORDER BY c.ano""")
display(anual)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(mensal["ano"].astype(str), mensal["liquido"] / 1e6, color=["#c9b79c", "#a3865a", "#5b4a2f"])
axes[0].set_title("Agosto: vendas líquidas (R$ mi)")
axes[1].plot(semestral["ano"].astype(str), semestral["conversao_pct"], marker="o", color="#5b4a2f")
axes[1].set_ylim(0, 60); axes[1].set_title("Jan-jul: conversão sobre fechados (%)")
axes[2].bar(anual["ano"].astype(str), anual["liquido"] / 1e6, color=["#c9b79c", "#a3865a", "#5b4a2f"])
axes[2].set_title("Anual: vendas líquidas (R$ mi)")
for ax in axes:
    for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()


,ano,vendas,liquido,comissoes
0,2024,57,5979748.72,128421.36
1,2025,83,10690598.33,308340.80
2,2026,146,20098339.23,544517.12


,ano,orcamentos,ganhos,perdidos,abertos,conversao_pct
0,2024,1391,331,1060,0,23.8
1,2025,1931,585,1340,6,30.4
2,2026,2102,776,853,473,47.6


,ano,vendas,liquido,bruto,margem_liquida,margem_sobre_custo,comissoes
0,2023,419,4.699141e+07,5.295660e+07,0.887,-66.353,1082979.11
1,2024,629,7.085745e+07,7.946770e+07,0.892,-87.262,1788805.84
2,2025,1039,1.309223e+08,1.473046e+08,0.889,-54.003,3515058.64


/tmp/ipykernel_107409/3581371732.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 3. Premiação: vendedores com faixa do SLA, parceiros e a carteira aberta no tempo

In [14]:
# Vendedores em 2026: vendas, conversão e faixa do SLA (base da premiação)
rank = q("""
WITH t AS (
  SELECT d.nome AS vendedor, COUNT(*) AS orcamentos, SUM(o.fl_ganho) AS ganhos,
         100.0 * SUM(o.fl_ganho) / NULLIF(SUM(o.fl_fechado), 0) AS conversao
  FROM dw.ft_orcamento o JOIN dw.dim_vendedor d ON d.sk_vendedor = o.sk_vendedor
  WHERE o.ano = 2026 GROUP BY d.nome)
SELECT t.vendedor, t.orcamentos, t.ganhos, CAST(t.conversao AS decimal(5,1)) AS conversao_pct, s.faixa
FROM t JOIN dw.dim_faixa_sla s ON s.tipo = 'CONVERSAO' AND t.conversao > s.limite_inferior_exclusivo AND t.conversao <= s.limite_superior_inclusivo
ORDER BY t.ganhos DESC""")
display(rank)
# Parceiros: quem trouxe mais vendas em 2025 e 2026
parceiros = q("""
SELECT TOP 12 p.tipo, p.nome, SUM(v.qtd_vendas) AS vendas, SUM(v.vl_liquido) AS liquido, SUM(v.vl_comissao_parceiro) AS comissao_parceiro
FROM dw.ft_venda v JOIN dw.dim_parceiro p ON p.sk_parceiro = v.sk_parceiro
WHERE v.ano IN (2025, 2026) AND p.sk_parceiro > 0 GROUP BY p.tipo, p.nome ORDER BY liquido DESC""")
display(parceiros)
# Carteira aberta ao longo do tempo (posição de fim de mês)
carteira = q("""
SELECT c.ano_mes, SUM(p.qtd_orcamentos) AS abertos, SUM(p.vl_liquido) AS liquido_aberto
FROM dw.ft_funil_posicao p JOIN dw.dim_calendario c ON c.sk_data = p.sk_data_posicao
WHERE p.grupo_fase = 'ABERTO' GROUP BY c.ano_mes ORDER BY c.ano_mes""")
fig, ax = plt.subplots(figsize=(14, 3.6))
ax.plot(carteira["ano_mes"], carteira["abertos"], color="#5b4a2f")
ax.set_title("Orçamentos em aberto na posição de fim de mês (2021 a 2026)")
ax.set_xticks(carteira["ano_mes"][::6]); ax.tick_params(axis="x", rotation=45)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()


,vendedor,orcamentos,ganhos,conversao_pct,faixa
0,Helena Souza Toledo,232,84,52.2,Excelente
1,Bruno Barros Santos,191,82,59.4,Excelente
2,Lucas Cavalcanti Xavier,154,57,53.8,Excelente
3,Gabriela Nogueira Bittencourt,119,51,60.7,Excelente
4,André Cunha Dias,128,48,57.1,Excelente
5,Leonardo Pereira Carvalho,133,46,51.1,Excelente
6,Thiago Cardoso Pinto,152,45,43.7,Excelente
7,Daniel Garcia Bittencourt,138,42,44.2,Excelente
8,Ricardo Teixeira Fonseca,135,41,41.4,Excelente
9,Renato Machado Prado,91,39,56.5,Excelente


,tipo,nome,vendas,liquido,comissao_parceiro
0,Escritório de arquitetura,Casa Prado,74,10739009.50,612123.54
1,Escritório de arquitetura,Soares Arquitetura,31,4862949.58,284968.83
2,Escritório de arquitetura,Luz Casa Arquitetura,35,3955061.65,218319.38
3,Escritório de arquitetura,Studio Nunes,23,3646444.71,218786.68
4,Escritório de arquitetura,Prado Arquitetura,17,3522874.53,194634.02
5,Escritório de arquitetura,Sampaio Arquitetura,34,2891080.53,184487.75
6,Construtora,Vasconcelos Incorporadora,28,2857275.62,57145.50
7,Escritório de arquitetura,Martins Arquitetura,5,2542032.21,177942.26
8,Escritório de arquitetura,Eixo Cunha,16,2517505.48,116308.76
9,Escritório de arquitetura,Batista Arquitetura,10,2113327.58,112863.10


/tmp/ipykernel_107409/4233095109.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Encerramento

O pipeline fechou o ciclo: do sistema comercial simulado ao warehouse multidimensional, com cada camada prestando contas à anterior. As perguntas da diretoria têm resposta por construção, e a sazonalidade, o arco de conversão e a carteira aberta contam a história da empresa.